In [5]:
from ultralytics import YOLO

# 1. Загружаем модель (используем treedetect.pt из вашей папки)
model = YOLO('treedetect.pt')

# 2. Основная информация о модели
print("--- Основная информация ---")
print(f"Тип модели: {model.task}")  # Должно быть 'detect'
print(f"Названия классов: {model.names}") 
print(f"Количество классов: {len(model.names)}")

# 3. Детальные параметры (архитектура и веса)
print("\n--- Детали архитектуры ---")
# Показывает количество параметров и слоев
model.info() 

# 4. Если хотите увидеть более подробную структуру слоев (опционально)
# print(model.model)

--- Основная информация ---
Тип модели: detect
Названия классов: {0: 'Tree'}
Количество классов: 1

--- Детали архитектуры ---
Model summary: 129 layers, 3,011,043 parameters, 0 gradients, 8.2 GFLOPs


(129, 3011043, 0, 8.1941504)

In [10]:
from ultralytics import YOLO
import cv2

# 1. Загружаем вашу модель
# Убедитесь, что файл treedetect.pt находится в той же папке, что и этот скрипт
model = YOLO('treedetect.pt')

# 2. Выполняем предсказание на изображении tree1.png
# conf=0.25 — порог уверенности (0.25 означает, что модель покажет объекты, в которых уверена на 25%+)
results = model.predict(source='tree4.jpg', conf=0.25, save=True)

# 3. Отображение результатов
for result in results:
    # Показываем изображение с нарисованными рамками (boxes)
    # result.show() — откроет стандартный просмотрщик картинок вашего ПК
    result.show()
    
    # Печатаем в консоль координаты найденных деревьев
    print("Обнаруженные объекты:", result.boxes.cls)
    print("Координаты (xyxy):", result.boxes.xyxy)


image 1/1 c:\a_awork\tree\tree4.jpg: 640x416 1 Tree, 78.8ms
Speed: 3.2ms preprocess, 78.8ms inference, 0.9ms postprocess per image at shape (1, 3, 640, 416)
Results saved to C:\a_awork\tree\runs\detect\predict4
Обнаруженные объекты: tensor([0.])
Координаты (xyxy): tensor([[124.7188, 278.4445, 712.8594, 836.2626]])


In [11]:
from ultralytics import YOLO
import cv2

# 1. Загружаем модель
model = YOLO('treedetect.pt')

# 2. Укажите ваш URL (добавьте /video для стандартных Android IP камер)
video_url = "http://10.73.48.78:8080/video" 

# Открываем видеопоток
cap = cv2.VideoCapture(video_url)

if not cap.isOpened():
    print("Ошибка: Не удалось подключиться к камере. Проверьте URL и сеть.")
else:
    print("Подключение установлено. Нажмите 'q' для выхода.")

while cap.isOpened():
    success, frame = cap.read()
    
    if success:
        # Запуск распознавания на текущем кадре
        # stream=True позволяет эффективнее работать с памятью на видео
        results = model.predict(frame, conf=0.4, imgsz=640, stream=True)
        
        for r in results:
            annotated_frame = r.plot() # Рисуем рамки на кадре
            
            # Логика: если есть объекты в кадре
            if len(r.boxes) > 0:
                cv2.putText(annotated_frame, "TREE DETECTED!", (50, 50), 
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

            # Показываем результат в окне
            cv2.imshow("YOLOv8 Real-time Tree Detection", annotated_frame)

        # Выход из цикла при нажатии клавиши 'q'
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    else:
        break

# Освобождаем ресурсы
cap.release()
cv2.destroyAllWindows()

Ошибка: Не удалось подключиться к камере. Проверьте URL и сеть.


In [16]:
import cv2
from ultralytics import YOLO

# 1. Загрузка модели
model = YOLO('treedetect.pt')

# 2. Настройка размеров (Установите нужные вам значения)
DISPLAY_WIDTH = 800  # Ширина окна
DISPLAY_HEIGHT = 600 # Высота окна

video_url = "http://10.135.1.10:8080/video"
cap = cv2.VideoCapture(video_url)
cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)

if not cap.isOpened():
    print("Ошибка: Не удалось подключиться.")
else:
    print(f"Камера подключена! Размер окна: {DISPLAY_WIDTH}x{DISPLAY_HEIGHT}")

while True:
    success, frame = cap.read()
    if not success:
        break

    # 3. Распознавание (imgsz=640 — стандарт, можно поставить 320 для скорости)
    results = model.predict(frame, conf=0.3, stream=True, imgsz=640)

    for r in results:
        annotated_frame = r.plot()
        
        # --- ИЗМЕНЕНИЕ РАЗМЕРА ТУТ ---
        # Мы меняем размер уже отрисованного кадра перед показом
        resized_frame = cv2.resize(annotated_frame, (DISPLAY_WIDTH, DISPLAY_HEIGHT))
        
        # Добавляем текст на уже измененный по размеру кадр
        status_text = "TREE DETECTED" if len(r.boxes) > 0 else "NO TREES"
        color = (0, 255, 0) if len(r.boxes) > 0 else (0, 0, 255)
        
        cv2.putText(resized_frame, f"STATUS: {status_text}", (20, 40), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

        # 4. Показываем измененный кадр
        cv2.imshow("YOLO Real-time: Tree Search", resized_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Камера подключена! Размер окна: 800x600

0: 640x640 (no detections), 92.0ms
Speed: 13.7ms preprocess, 92.0ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 92.9ms
Speed: 7.5ms preprocess, 92.9ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 81.3ms
Speed: 6.3ms preprocess, 81.3ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 72.5ms
Speed: 6.5ms preprocess, 72.5ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 73.4ms
Speed: 5.7ms preprocess, 73.4ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 70.3ms
Speed: 5.4ms preprocess, 70.3ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 75.2ms
Speed: 5.2ms preprocess, 75.2ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections),